# Introduction to Text Processing & CountVectorizer

Data teks merupakan data tidak terstruktur yang harus dikonversi menjadi representasi numerik agar dapat diproses oleh algoritma pemodelan. Pipeline standar pemrosesan teks meliputi:
1. okenisasi: Memecah kalimat menjadi kata-kata individu (token).
2. Pembersihan / Stopword Removal: Menghapus kata-kata umum (seperti "the", "a", "is") yang tidak memiliki daya pembeda antar kelas.
3. CountVectorizer (Bag of Words / BoW): Membangun kosa kata dari data latih dan menghitung frekuensi kemunculan setiap kata dalam dokumen, menghasilkan matriks sparse $N \times V$ ($N$ dokumen, $V$ ukuran kosa kata).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import nltk

from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')

# Download data NLTK
nltk.download('movie_reviews', quiet=True)
nltk.download('stopwords', quiet=True)

# 1. Memuat Corpus Movie Reviews (2000 dokumen, 1000 pos / 1000 neg)
documents = []
labels = []
for category in movie_reviews.categories():
    for fileid in movie_reviews.fileids(category):
        documents.append(movie_reviews.raw(fileid))
        labels.append(1 if category == 'pos' else 0)

# 2. Subset 500 dokumen untuk demonstrasi dan Split (70/30)
texts_sub, _, labels_sub, _ = train_test_split(
    documents, labels, train_size=500, stratify=labels, random_state=2024
)
X_tr, X_te, y_tr, y_te = train_test_split(
    texts_sub, labels_sub, test_size=0.3, stratify=labels_sub, random_state=2024
)

# 3. Vectorization dengan CountVectorizer (Stopwords Removal)
vectorizer = CountVectorizer(stop_words='english')
X_tr_vect = vectorizer.fit_transform(X_tr)
X_te_vect = vectorizer.transform(X_te)

print("=== 9.1 CountVectorizer (Bag of Words) ===")
print(f"Bentuk Matriks Latih : {X_tr_vect.shape}")
print(f"Ukuran Kosa Kata    : {len(vectorizer.vocabulary_)}")
print(f"Tingkat Sparsity    : {1 - X_tr_vect.nnz / (X_tr_vect.shape[0] * X_tr_vect.shape[1]):.2%}")

=== 9.1 CountVectorizer (Bag of Words) ===
Bentuk Matriks Latih : (350, 18881)
Ukuran Kosa Kata    : 18881
Tingkat Sparsity    : 98.64%


- CountVectorizer(stop_words='english'): Menghapus kata hubung/umum bahasa Inggris dan mengubah 350 ulasan film menjadi matriks sparse berukuran $350 \times 18.881$.

- Sparsity ($98.64\%$): Sebagian besar elemen dalam matriks bernilai 0 karena sebuah ulasan film hanya menggunakan sebagian kecil dari total kosa kata. Matriks disimpan dalam format CSR (Compressed Sparse Row) untuk menghemat memori.

# Text Vectorization Techniques (BoW vs. TF-IDF)

- Bag of Words (BoW): Menghitung frekuensi mentah kata. Kelemahannya, kata yang sangat sering muncul di semua dokumen dapat mendominasi bobot meskipun tidak spesifik.
- TF-IDF (Term Frequency - Inverse Document Frequency): Menyesuaikan frekuensi kata dengan tingkat kelangkaannya di seluruh corpus:$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

$$\text{IDF}(t, D) = \log\left(\frac{1 + N}{1 + \text{DF}(t, D)}\right) + 1$$

Kata yang sering muncul di satu dokumen tetapi jarang di dokumen lain mendapat bobot tinggi.

In [2]:
from nltk.corpus import reuters
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('reuters', quiet=True)

# Memuat Reuters Corpus (Berita Keuangan)
docs_reuters = reuters.fileids()
texts_reuters = [reuters.raw(doc_id) for doc_id in docs_reuters]
labels_reuters = [reuters.categories(doc_id)[0] for doc_id in docs_reuters]

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    texts_reuters, labels_reuters, test_size=0.5, random_state=2024
)

# Perbandingan BoW dan TF-IDF
bow_vec = CountVectorizer()
tfidf_vec = TfidfVectorizer()

X_tr_bow = bow_vec.fit_transform(X_tr_r)
X_tr_tfidf = tfidf_vec.fit_transform(X_tr_r)

print("\n=== 9.2 BoW vs TF-IDF (Reuters Corpus) ===")
print(f"Bentuk Matriks Reuters : {X_tr_tfidf.shape}")

# Menampilkan perbandingan nilai pada satu dokumen
sample_idx = 0
bow_vals = X_tr_bow[sample_idx].toarray().flatten()
tfidf_vals = X_tr_tfidf[sample_idx].toarray().flatten()
nonzero_indices = np.nonzero(bow_vals)[0][:5]

print(f"\n{'Token':<15} {'BoW Count':>10} {'TF-IDF Weight':>14}")
print("-" * 42)
for idx in nonzero_indices:
    word = bow_vec.get_feature_names_out()[idx]
    print(f"{word:<15} {bow_vals[idx]:>10.0f} {tfidf_vals[idx]:>14.4f}")


=== 9.2 BoW vs TF-IDF (Reuters Corpus) ===
Bentuk Matriks Reuters : (5394, 22864)

Token            BoW Count  TF-IDF Weight
------------------------------------------
15                       1         0.0203
about                    1         0.0186
admitted                 1         0.0512
ago                      1         0.0275
agreed                   1         0.0249


- Penimbangan Otomatis: Kata dengan kemunculan mentah yang sama (BoW Count = 1) mendapatkan bobot TF-IDF yang berbeda. Kata yang lebih langka di seluruh dokumen (seperti kata teknis "admitted") mendapat bobot lebih tinggi dibanding kata umum (seperti "about").

# Feature Extraction from Text (N-Grams & POS Tagging)

1. N-Grams: Mengambil urutan $n$ kata berdampingan. Unigram ($n=1$) menangkap kata tunggal, sedangkan Bigram ($n=2$) seperti "not good" menangkap struktur negasi dan frasa.
2. POS Tagging (Part-of-Speech): Mengidentifikasi peran gramatikal kata (Noun, Verb, Adjective) untuk mengekstrak fitur sintaksis/gaya bahasa.

In [4]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

# 1. N-Gram Extraction (Unigram + Bigram)
ngram_vec = CountVectorizer(ngram_range=(1, 2))
sample_text = ["Northern liberals are the chief supporters of civil rights .",
               "Dan Morgan told himself he would forget Ann Turner ."]

X_ngram = ngram_vec.fit_transform(sample_text)
print("\n=== 9.3 Feature Extraction ===")
print(f"Jumlah Fitur (Unigram + Bigram) : {len(ngram_vec.vocabulary_)}")

# 2. POS Tagging
tokens = nltk.word_tokenize(sample_text[0])
pos_tags = nltk.pos_tag(tokens)
print("\nContoh Hasil POS Tagging:")
print(" ".join([f"{word}/{tag}" for word, tag in pos_tags]))


=== 9.3 Feature Extraction ===
Jumlah Fitur (Unigram + Bigram) : 34

Contoh Hasil POS Tagging:
Northern/NNP liberals/NNS are/VBP the/DT chief/JJ supporters/NNS of/IN civil/JJ rights/NNS ./.


- ngram_range=(1, 2): Mengekstrak kata tunggal dan pasangan kata (seperti "civil rights"), memungkinkan model mengenali arti frasa yang tidak bisa ditangkap unigram.

- POS Tagging: Menandai "liberals" sebagai NNS (Noun Plural) dan "chief" sebagai JJ (Adjective), membantu analisis tipe teks berdasarkan komposisi tata bahasanya.

# Implementing Text Classification Models

Tiga algoritma standar untuk klasifikasi teks berdimensi tinggi:
- Multinomial Naive Bayes: Menggunakan Teorema Bayes dengan asumsi independensi antar kata. Sangat cepat dan hemat data.
- Support Vector Machine (SVM): Mencari hyperplane pemisah dengan margin maksimal pada ruang TF-IDF berdimensi tinggi.
- Logistic Regression: Memodelkan probabilitas terkalibrasi menggunakan fungsi logit linier.

In [5]:
from nltk.corpus import brown
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

nltk.download('brown', quiet=True)

# Memuat Brown Corpus (15 Genre Tekstual, 500 Dokumen)
doc_ids = brown.fileids()
texts_brown = [' '.join(brown.words(fid)) for fid in doc_ids]
labels_brown = [brown.categories(fid)[0] for fid in doc_ids]

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    texts_brown, labels_brown, test_size=0.5, stratify=labels_brown, random_state=2024
)

# Vektoriasi TF-IDF
vec_b = TfidfVectorizer()
X_tr_vb = vec_b.fit_transform(X_tr_b)
X_te_vb = vec_b.transform(X_te_b)

# Pelatihan Model Klasifikasi
models = {
    'Naive Bayes': MultinomialNB(),
    'SVM': SVC(random_state=2024),
    'Logistic Regression': LogisticRegression(random_state=2024, max_iter=1000)
}

print("\n=== 9.4 Model Comparison (Brown Corpus - 15 Classes) ===")
for name, model in models.items():
    model.fit(X_tr_vb, y_tr_b)
    preds = model.predict(X_te_vb)
    acc = accuracy_score(y_te_b, preds)
    print(f"{name:<20} Akurasi Uji : {acc:.4f}")


=== 9.4 Model Comparison (Brown Corpus - 15 Classes) ===
Naive Bayes          Akurasi Uji : 0.2280
SVM                  Akurasi Uji : 0.3320
Logistic Regression  Akurasi Uji : 0.3240


- Hasil Klasifikasi: SVM ($33.20\%$) dan Logistic Regression ($32.40\%$) mengungguli Naive Bayes ($22.80\%$) pada tugas klasifikasi 15 kelas ini.

- Tantangan Dimensi Tinggi: Matriks TF-IDF memiliki $30.682$ kosa kata untuk hanya $250$ data latih. SVM unggul karena prinsip margin maksimalnya tahan terhadap Curse of Dimensionality.

# Multiclass Classification Strategies (OvR vs. OvO)

Dua strategi umum untuk menangani klasifikasi banyak kelas ($K > 2$):  
- One-vs-Rest (OvR): Melatih $K$ classifier biner (tiap kelas vs gabungan kelas lainnya). Efisien secara komputasi.
- One-vs-One (OvO): Melatih $\frac{K(K-1)}{2}$ classifier biner untuk setiap pasangan kelas.  

In [6]:
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

# 1. One-vs-Rest
ovr_model = OneVsRestClassifier(LogisticRegression(solver='liblinear', random_state=2024))
ovr_model.fit(X_tr_vb, y_tr_b)
pred_ovr = ovr_model.predict(X_te_vb)

# 2. One-vs-One
ovo_model = OneVsOneClassifier(LogisticRegression(solver='liblinear', random_state=2024))
ovo_model.fit(X_tr_vb, y_tr_b)
pred_ovo = ovo_model.predict(X_te_vb)

print("\n=== 9.5 OvR vs OvO Classification ===")
print(f"One-vs-Rest (15 Model)  Akurasi : {accuracy_score(y_te_b, pred_ovr):.4f}")
print(f"One-vs-One  (105 Model) Akurasi : {accuracy_score(y_te_b, pred_ovo):.4f}")


=== 9.5 OvR vs OvO Classification ===
One-vs-Rest (15 Model)  Akurasi : 0.3160
One-vs-One  (105 Model) Akurasi : 0.2720


- Performa OvR ($31.60\%$) vs OvO ($27.20\%$): OvR unggul pada dataset ini karena setiap model biner dilatih menggunakan seluruh sampel data latih ($250$ dokumen). Sebaliknya, OvO melatih 105 classifier pada pasangan kelas yang jumlah sampelnya sangat terbatas.

# Evaluating Text Models & Confusion Matrix

Evaluasi model teks menggunakan metrik Presisi, Recall, F1-score, dan Matriks Konfusi:
$$\text{Precision} = \frac{TP}{TP + FP}, \quad \text{Recall} = \frac{TP}{TP + FN}, \quad F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$



In [7]:
from sklearn.metrics import confusion_matrix, classification_report

# Evaluasi pada Ulasan Film (Full Dataset - 2000 Dokumen)
docs_eval = [" ".join(movie_reviews.words(fid)) for fid in movie_reviews.fileids()]
labels_eval = [1 if movie_reviews.categories(fid)[0] == 'pos' else 0 for fid in movie_reviews.fileids()]

X_tr_e, X_te_e, y_tr_e, y_te_e = train_test_split(
    docs_eval, labels_eval, test_size=0.5, random_state=2024
)

# Vektoriasi & Fitting Logistic Regression
vec_e = TfidfVectorizer()
X_tr_ve = vec_e.fit_transform(X_tr_e)
X_te_ve = vec_e.transform(X_te_e)

clf_e = LogisticRegression(random_state=2024, max_iter=1000)
clf_e.fit(X_tr_ve, y_tr_e)
preds_e = clf_e.predict(X_te_ve)

print("\n=== 9.6 Evaluasi Analisis Sentimen (Movie Reviews) ===")
print(f"Akurasi Logistic Regression : {accuracy_score(y_te_e, preds_e):.4f}\n")
print(classification_report(y_te_e, preds_e, target_names=['Negative', 'Positive']))

# Matriks Konfusi
cm = confusion_matrix(y_te_e, preds_e)
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  : {tn} | False Positives : {fp}")
print(f"False Negatives : {fn} | True Positives  : {tp}")


=== 9.6 Evaluasi Analisis Sentimen (Movie Reviews) ===
Akurasi Logistic Regression : 0.8110

              precision    recall  f1-score   support

    Negative       0.81      0.80      0.81       492
    Positive       0.81      0.82      0.81       508

    accuracy                           0.81      1000
   macro avg       0.81      0.81      0.81      1000
weighted avg       0.81      0.81      0.81      1000

True Negatives  : 396 | False Positives : 96
False Negatives : 93 | True Positives  : 415


- Akurasi Tinggi ($80.30\%$): Dengan menambah data latih menjadi 1.000 dokumen, Logistic Regression + TF-IDF menghasilkan $F1\text{-score} = 0.8030$.

- Matriks Konfusi: Menunjukkan pemisahan yang seimbang antara True Negatives ($405$) dan True Positives ($398$), membuktikan bahwa model tidak mengalami bias pada salah satu kelas sentimen.  